# Feature Engineering

In [2]:
import numpy as np
import pandas as pd
from pathlib import Path
import sys

root = Path.cwd().resolve()
for parent in [root] + list(root.parents):
    if (parent / "ml_engine").exists():
        sys.path.insert(0, str(parent))
        break
else:
    sys.path.insert(0, str(root))

from ml_engine.preprocessing import load_data, clean_beneficiary_data, clean_inpatient_data, clean_outpatient_data, merge_claims, merge_beneficiary
from ml_engine.feature_engineering import extract_top_codes, create_code_indicators, create_claim_features, aggregate_provider_features, add_labels, save_provider_master

In [3]:
benf_df, inpatient_df, outpatient_df, labels_df = load_data(
    '../data/Train_Beneficiarydata-1542865627584.csv',
    '../data/Train_Inpatientdata-1542865627584.csv',
    '../data/Train_Outpatientdata-1542865627584.csv',
    '../data/Train-1542865627584.csv'
)

In [4]:
inpatient_df = clean_inpatient_data(inpatient_df)
outpatient_df = clean_outpatient_data(outpatient_df)
benf_df = clean_beneficiary_data(benf_df, inpatient_df, outpatient_df)

In [5]:
claims = merge_claims(inpatient_df, outpatient_df)
claims_bene = merge_beneficiary(claims, benf_df)
print(f"Total claims after union: {len(claims):,}")
print(claims["ClaimType"].value_counts())

D:\GitHub\Fraud-claim-detection\ml_engine\preprocessing.py:125: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  claims = pd.concat([inpatient_df, outpatient_df], ignore_index=True, sort=False)


Total claims after union: 558,211
ClaimType
Outpatient    517737
Inpatient      40474
Name: count, dtype: int64


In [6]:
claims_bene = create_claim_features(claims_bene)

In [7]:
print("Extracting top-100 diagnosis and procedure codes ...")

top100_diag, top100_proc = extract_top_codes(claims_bene, top_n=100)
print(f"  Top-100 diagnosis codes selected: {len(top100_diag)}")
print(f"  Top-100 procedure codes selected: {len(top100_proc)}")

Extracting top-100 diagnosis and procedure codes ...
  Top-100 diagnosis codes selected: 100
  Top-100 procedure codes selected: 100


In [8]:
claims_bene = create_code_indicators(claims_bene, top100_diag, top100_proc)
print("Code indicator columns created")

D:\GitHub\Fraud-claim-detection\ml_engine\feature_engineering.py:26: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  claims_bene[f"Diag_{code}"] = (claims_bene["ClmDiagnosisCode_1"] == code).astype(int)
D:\GitHub\Fraud-claim-detection\ml_engine\feature_engineering.py:26: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  claims_bene[f"Diag_{code}"] = (claims_bene["ClmDiagnosisCode_1"] == code).astype(int)
D:\GitHub\Fraud-claim-detection\ml_engine\feature_engineering.py:26: PerformanceWarning: DataFrame is highly fragmented.  This is us

Code indicator columns created


D:\GitHub\Fraud-claim-detection\ml_engine\feature_engineering.py:29: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  claims_bene[f"Proc_{code}"] = (claims_bene["ClmProcedureCode_1"] == code).astype(int)
D:\GitHub\Fraud-claim-detection\ml_engine\feature_engineering.py:29: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  claims_bene[f"Proc_{code}"] = (claims_bene["ClmProcedureCode_1"] == code).astype(int)


In [9]:
provider_df = aggregate_provider_features(claims_bene, top100_diag, top100_proc)
print(f"✓ Provider-level DataFrame: {provider_df.shape}")

✓ Provider-level DataFrame: (5410, 228)


In [10]:
labels_df["FraudLabel"] = (labels_df["PotentialFraud"] == "Yes").astype(int)
final_df = add_labels(provider_df, labels_df)
print("Final DataFrame shape :", final_df.shape)
print("Fraud label distribution:")
print(final_df["FraudLabel"].value_counts())
print(final_df["FraudLabel"].value_counts(normalize=True).round(3))

Final DataFrame shape : (5410, 230)
Fraud label distribution:
FraudLabel
0    4904
1     506
Name: count, dtype: int64
FraudLabel
0    0.906
1    0.094
Name: proportion, dtype: float64


In [11]:
null_final = final_df.isnull().sum()
null_final = null_final[null_final > 0]
null_pct_f = (null_final / len(final_df) * 100).round(2)

if len(null_final) == 0:
    print("✓ No missing values in final provider-level DataFrame!")
else:
    print("Remaining missing values:")
    print(pd.DataFrame({"Missing": null_final, "Missing%": null_pct_f}))

✓ No missing values in final provider-level DataFrame!


In [12]:
CORE_FEATURES = [
    "TotalClaims", "TotalReimbursed", "AvgReimbursed",
    "AvgClaimDuration", "AvgDaysInHospital",
    "UniquePatients", "UniqueAttendPhys",
    "SameAttendOperRate", "AvgChronicConds", "InpatientRatio",
]

In [13]:
profile = (
    final_df
    .groupby("FraudLabel")[CORE_FEATURES]
    .median()
    .T
    .rename(columns={0: "Not Fraud (median)", 1: "Fraud (median)"})
)
profile["Ratio (Fraud/Not Fraud)"] = (
    profile["Fraud (median)"] / profile["Not Fraud (median)"].replace(0, np.nan)
).round(2)

print("=" * 65)
print("FRAUD PROVIDER PROFILE  —  Median Feature Comparison")
print("=" * 65)
print(profile.to_string())

FRAUD PROVIDER PROFILE  —  Median Feature Comparison
FraudLabel          Not Fraud (median)  Fraud (median)  Ratio (Fraud/Not Fraud)
TotalClaims                  27.000000      155.500000                     5.76
TotalReimbursed           15055.000000   373450.000000                    24.81
AvgReimbursed               332.192088     2576.480084                     7.76
AvgClaimDuration              1.512224        2.478220                     1.64
AvgDaysInHospital             0.000000        1.361804                      NaN
UniquePatients               22.000000      117.000000                     5.32
UniqueAttendPhys              5.000000       23.000000                     4.60
SameAttendOperRate            0.111111        0.093780                     0.84
AvgChronicConds               4.500000        4.716257                     1.05
InpatientRatio                0.000000        0.217677                      NaN


In [14]:
save_provider_master(final_df, "provider_master.csv")
print("Saved provider_master.csv")

Saved provider_master.csv
